# Batch Radiomics Feature Extraction
Extract texture and intensity features from all 134 ultrasound images using PyRadiomics.
Each image is paired with its eroded binary mask from the preprocessing pipeline.
Output: a single CSV with one row per study and ~100 feature columns.

In [1]:
import os
import cv2
import pydicom
import numpy as np
import pandas as pd
import SimpleITK as sitk
from radiomics import featureextractor

print("Imports OK")

Imports OK


In [2]:
# paths
raw_data_folder = os.path.join("..", "data", "PANCREAS_2", "PANCREAS_2")
mask_folder = os.path.join("..", "data", "PANCREAS_PREPROCESSED_CONTOUR_SUBTRACTED_ERODED_K3_I1", "masks")
manifest_path = os.path.join("..", "data", "PANCREAS_PREPROCESSED_CONTOUR_SUBTRACTED_ERODED_K3_I1", "manifest_eroded_CONTOUR_SUBTRACTED_k3_i1.csv")
output_csv_path = os.path.join("reports", "12_radiomics_features_k3_i1.csv")

# load manifest and check for zero-pixel masks
manifest = pd.read_csv(manifest_path)
zero_mask_rows = manifest[manifest["eroded_pixels"] == 0]
print(f"Manifest loaded: {len(manifest)} studies")
if len(zero_mask_rows) > 0:
    print(f"WARNING: {len(zero_mask_rows)} studies have zero pixels in mask:")
    print(zero_mask_rows["study_id"].tolist())
else:
    print("All masks have nonzero pixels.")

Manifest loaded: 134 studies
All masks have nonzero pixels.


In [3]:
def find_dicom_path(study_id):
    """Find the DICOM file path for a given study ID."""
    patient_folder = os.path.join(raw_data_folder, study_id)
    if not os.path.isdir(patient_folder):
        return None

    # find date subfolder (skip hidden files)
    subfolders = [f for f in os.listdir(patient_folder) if not f.startswith(".")]
    if len(subfolders) == 0:
        return None

    date_folder = os.path.join(patient_folder, subfolders[0])

    # find the DICOM file
    files = [f for f in os.listdir(date_folder) if not f.startswith(".")]
    if len(files) == 0:
        return None

    return os.path.join(date_folder, files[0])

In [4]:
def load_image_and_mask(study_id):
    """Load DICOM as grayscale + mask as binary, return as SimpleITK objects."""
    # load DICOM
    dicom_path = find_dicom_path(study_id)
    if dicom_path is None:
        raise FileNotFoundError(f"No DICOM found for {study_id}")

    ds = pydicom.dcmread(dicom_path)
    pixels = ds.pixel_array

    if len(pixels.shape) == 3:
        gray = cv2.cvtColor(pixels, cv2.COLOR_RGB2GRAY)
    else:
        gray = pixels

    # load mask
    mask_file = study_id + "_mask_eroded_k3_i1.png"
    mask_path = os.path.join(mask_folder, mask_file)
    mask_raw = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask_raw is None:
        raise FileNotFoundError(f"Mask not found: {mask_path}")

    # binarize: 255 -> 1
    mask_binary = np.zeros_like(mask_raw, dtype=np.uint8)
    mask_binary[mask_raw > 0] = 1

    # handle shape mismatch
    if gray.shape != mask_binary.shape:
        mask_binary = cv2.resize(mask_binary, (gray.shape[1], gray.shape[0]),
                                 interpolation=cv2.INTER_NEAREST)

    # cast to int16 -- PyRadiomics texture features can fail on uint8
    gray = gray.astype(np.int16)

    # convert to SimpleITK
    sitk_image = sitk.GetImageFromArray(gray)
    sitk_mask = sitk.GetImageFromArray(mask_binary)

    return sitk_image, sitk_mask

In [5]:
# configure the extractor
# texture + intensity features only, no shape (per supervisor guidance)
# force2D because our images are 2D slices -- without this, some masks cause
# GLCM/GLRLM failures when PyRadiomics tries 3D co-occurrence on a depth=1 volume
settings = {"force2D": True, "force2Ddimension": 0}
extractor = featureextractor.RadiomicsFeatureExtractor(**settings)
extractor.disableAllFeatures()
extractor.enableFeatureClassByName("firstorder")
extractor.enableFeatureClassByName("glcm")
extractor.enableFeatureClassByName("glrlm")
extractor.enableFeatureClassByName("glszm")
extractor.enableFeatureClassByName("gldm")
extractor.enableFeatureClassByName("ngtdm")

print("Extractor configured.")
print("Enabled feature classes:", extractor.enabledFeatures)

Extractor configured.
Enabled feature classes: {'firstorder': [], 'glcm': [], 'glrlm': [], 'glszm': [], 'gldm': [], 'ngtdm': []}


In [6]:
# run extraction on all studies
results = []
errors = []
study_ids = manifest["study_id"].tolist()

for i, study_id in enumerate(study_ids):
    try:
        sitk_image, sitk_mask = load_image_and_mask(study_id)
        features = extractor.execute(sitk_image, sitk_mask)

        # keep only actual features (skip diagnostics_ metadata)
        row = {"study_id": study_id}
        for key, value in features.items():
            if not key.startswith("diagnostics_"):
                row[key] = float(value)

        results.append(row)

    except Exception as e:
        errors.append({"study_id": study_id, "error": str(e)})
        print(f"  ERROR on {study_id}: {e}")

    if (i + 1) % 10 == 0:
        print(f"  processed {i + 1}/{len(study_ids)}")

print(f"\nDone. {len(results)} succeeded, {len(errors)} failed.")
if errors:
    print("Failed studies:")
    for err in errors:
        print(f"  {err['study_id']}: {err['error']}")

GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Avera

  processed 10/134


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


  processed 20/134


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


...

  processed 120/134


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated
GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated


  processed 130/134


GLCM is symmetrical, therefore Sum Average = 2 * Joint Average, only 1 needs to be calculated



Done. 134 succeeded, 0 failed.


In [7]:
# build dataframe and save
df = pd.DataFrame(results)
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")

# list feature classes found
feature_cols = [c for c in df.columns if c != "study_id"]
print(f"Feature columns: {len(feature_cols)}")

# save
os.makedirs("reports", exist_ok=True)
df.to_csv(output_csv_path, index=False)
print(f"Saved to {output_csv_path}")

df.head()

Shape: 134 rows x 94 columns
Feature columns: 93
Saved to reports/12_radiomics_features_k3_i1.csv


,study_id,original_firstorder_10Percentile,original_firstorder_90Percentile,original_firstorder_Energy,original_firstorder_Entropy,original_firstorder_InterquartileRange,original_firstorder_Kurtosis,original_firstorder_Maximum,original_firstorder_MeanAbsoluteDeviation,original_firstorder_Mean,...,original_gldm_LargeDependenceLowGrayLevelEmphasis,original_gldm_LowGrayLevelEmphasis,original_gldm_SmallDependenceEmphasis,original_gldm_SmallDependenceHighGrayLevelEmphasis,original_gldm_SmallDependenceLowGrayLevelEmphasis,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength
0,01_01,26.0,65.0,40347446.0,1.447682,21.0,2.977687,106.0,12.202259,44.459479,...,2.076722,0.260298,0.139974,0.814440,0.037863,46.077809,0.001728,0.947164,0.003742,0.018856
1,01_02,27.0,62.0,24721784.0,1.335817,18.0,3.545282,115.0,10.826002,43.988339,...,1.897141,0.253151,0.152729,0.859875,0.041468,46.923585,0.001628,1.503508,0.004876,0.019147
2,01_03,35.0,72.0,40855206.0,1.425460,20.0,4.058580,124.0,11.664058,52.668379,...,1.349964,0.177598,0.154562,1.150995,0.027980,47.690567,0.001422,1.391369,0.005705,0.014597
3,01_04,37.0,72.0,33992426.0,1.384659,18.0,3.724079,116.0,11.012528,54.465636,...,1.287580,0.166851,0.147384,1.133239,0.025525,30.302721,0.002060,1.195866,0.004488,0.021005
4,01_05,43.0,77.0,22557430.0,1.350046,16.0,3.953666,119.0,10.473772,58.905324,...,1.020789,0.139657,0.163403,1.391379,0.023950,22.158772,0.002675,1.584846,0.005742,0.026350
